In [ ]:
import time
start_time = time.time()
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
import spark_sdk as ss
import yaml
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import col, sum as sum_, count, max as max_, datediff, lit, min as min_, desc, when, unix_timestamp, row_number, struct, collect_list,\
to_date, countDistinct, exp, lit
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import pytz

import spark_sdk as ss
ss.PySpark(yarn=False, num_executors=48, driver_memory='32g', executor_memory='32g',
           add_on_config0=('spark.port.maxRetries', '1000'),
           add_on_config1=("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog"),
           add_on_config2=('spark.databricks.delta.retentionDurationCheck.enabled', 'false'),
           add_on_config3=('spark.sql.sources.partitionOverwriteMode', 'dynamic')
          )
spark = ss.PySpark().spark

from pyspark.sql.functions import to_date

full_dataset = spark.read.format('parquet').load('file:/home/baond12/projects/AI_Recommend_System_2025/BTL_HTTM/CS-recommendation-system/backend/full_dataset.parquet')\
.withColumn('event_date', to_date('TimePlaced'))

# Lấy danh sách 7 ngày cuối cùng có trong dữ liệu
max_date_row = full_dataset.agg(max_("event_date")).collect()[0][0]
# Tính toán ngày bắt đầu validation (cách max date 6 ngày)
val_date = max_date_row - timedelta(days=0)

start_train_date = val_date - timedelta(days=60)

end_train_date = val_date - timedelta(days=1)

print(f"Start Training Date: {start_train_date} -- End Training Date: {end_train_date}")
print(f"Validation Date: {val_date}")

train_data = full_dataset.filter(f"event_date <= '{end_train_date}'")
val_data = full_dataset.filter(f"event_date = '{val_date}'")
new_user_df = val_data.join(train_data, ['Account_Id'], 'left_anti').agg(countDistinct('Account_Id').alias('nb_new_users'))

print(f"Số lượng khách hàng mới: {new_user_df.collect()[0][0]}")

def item_feature_engineering(data_source):
    max_event_time_per_user = data_source.groupBy('Account_Id').agg(max_('event_date').alias('max_event_time_per_user'))
    ds_with_max_time = data_source.join(max_event_time_per_user, ['Account_Id'], 'inner')\
.select('event_date', 'Account_Id', 'ProductId', 'max_event_time_per_user')
    ds_with_max_time = ds_with_max_time.withColumn(
    "days_diff_to_max", datediff(col("max_event_time_per_user"), col("event_date"))
)

    lambda_decay = 0.05
    ds_with_max_time = ds_with_max_time.withColumn(
        "decayed_score", exp(-lit(lambda_decay) * col("days_diff_to_max"))
    )
    
    ds_rating = ds_with_max_time.groupBy("Account_Id", "ProductId")\
        .agg({"decayed_score": "sum"})\
        .withColumnRenamed("sum(decayed_score)", "importance_score")
    
    return ds_rating

train_feature_engineering = item_feature_engineering(train_data)

import time
start_time = time.time()

from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.sql.functions import col, explode

# user & item encoding
user_indexer = StringIndexer(inputCol="Account_Id", outputCol="user_index").setHandleInvalid("skip")
item_indexer = StringIndexer(inputCol="ProductId", outputCol="item_index").setHandleInvalid("skip")


user_encoder = user_indexer.fit(train_feature_engineering)
item_encoder = item_indexer.fit(train_feature_engineering)

train_feature_engineering = user_encoder.transform(train_feature_engineering)
train_feature_engineering = item_encoder.transform(train_feature_engineering)

user_index_to_id = train_feature_engineering.groupBy('Account_Id', 'user_index').agg(count('*').alias('count')).drop('count')
item_index_to_id = train_feature_engineering.groupBy('ProductId', 'item_index').agg(count('*').alias('count')).drop('count')

item_als_obj = ALS(
    userCol="user_index",
    itemCol="item_index",
    ratingCol="importance_score",
    implicitPrefs=True,  # Vì score của bạn là inferred signal, không phải rating rõ ràng
    nonnegative=True,
    coldStartStrategy="drop",
    rank=100,
    maxIter=10,
    regParam=0.1
)

item_als_model = item_als_obj.fit(train_feature_engineering)

user_item_recs = item_als_model.recommendForAllUsers(100)


user_item_recs_exploded = (user_item_recs
    .withColumn("rec_exp", explode(col("recommendations")))
    .select(
        col("user_index"),
        col("rec_exp.item_index").alias("item_index"),
        col("rec_exp.rating").alias("item_pred_score")
    )
)

user_item_recs_final = (user_item_recs_exploded
    .join(user_index_to_id, on="user_index", how="left")
    .join(item_index_to_id, on="item_index", how="left")
    .select("Account_Id", "ProductId", "item_pred_score")
)

end_time = time.time()

print(f"Duration: {end_time - start_time}")

user_item_recs_final = user_item_recs_final.withColumn('d', lit(val_date.strftime('%Y-%m-%d')))
user_item_recs_final.write.format('parquet')\
.mode('overwrite').option('overwriteSchema', 'true')\
.partitionBy(['d'])\
.save("file:/mnt/projects-data/recommendation/dev_folder/baond12/btl_httm/user_item_recs_final.parquet")